In [2]:
import pandas as pd
import numpy as np
import torch
from transformer_time_series_enc_dec import train_model,InformerForecaster,create_dataloaders,TrainConfig,inverse_transform,init_weights
import plotly.express as px
import matplotlib.pyplot as plt
import random

In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [4]:
import json
from datetime import datetime
import os

def create_loss_plot(train_hist, val_hist, steps_hist, total_params=None):
    """Create a loss plot from training history"""
    df_train = pd.DataFrame({
        "Step": steps_hist,
        "Loss": train_hist,
        "Type": "Train"
    })
    if val_hist:
        val_steps, val_losses = zip(*val_hist)
        df_val = pd.DataFrame({
            "Step": val_steps,
            "Loss": val_losses,
            "Type": "Validation"
        })
        df_loss = pd.concat([df_train, df_val], ignore_index=True)
    else:
        df_loss = df_train
        
    title = "Training and Validation Loss"
    if total_params is not None:
        title += f"\nTotal Parameters: {total_params:,}"
    
    fig = px.line(df_loss, x="Step", y="Loss", color="Type", title=title)
    return fig, df_loss

def create_prediction_plots(model, val_loader, asset_idx, scaler, config, num_batches=10, save_dir=None):
    """Create prediction plots and calculate metrics for validation batches
    
    Args:
        model: The trained model
        val_loader: Validation data loader
        asset_idx: Index of the asset to predict
        scaler: Scaler used for data normalization
        config: Model configuration
        num_batches: Number of batches to visualize
        save_dir: If provided, save plots to this directory
        
    Returns:
        list: List of dictionaries containing metrics for each batch
    """
    metrics = []
    device = next(model.parameters()).device
    
    with torch.no_grad():
        for i, batch in enumerate(val_loader):
            if i >= num_batches:
                break
                
            # Unpack the batch tuple correctly
            x, timestamps = batch
            x = x.to(device)
            timestamps = timestamps.to(device)
            y_pred = model(x, timestamps)
            y_true = x[:, -model.pred_len:, asset_idx]

            # Move to CPU and convert to numpy
            y_pred = y_pred.cpu().numpy()
            y_true = y_true.cpu().numpy()

            # Inverse transform to real prices
            y_pred_price = inverse_transform(y_pred.flatten(), scaler, asset_idx, config["d_input"])
            y_true_price = inverse_transform(y_true.flatten(), scaler, asset_idx, config["d_input"])

            # Create plots
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))

            # Normalised scale
            axes[0].plot(y_true.flatten(), label="Normalised True")
            axes[0].plot(y_pred.flatten(), label="Normalised Predicted")
            axes[0].set_title(f"Batch {i+1} – Normalised")
            axes[0].set_xlabel("Prediction Step")
            axes[0].set_ylabel("Scaled Value")
            axes[0].legend()

            # Real-price scale
            axes[1].plot(y_true_price, label="Real True")
            axes[1].plot(y_pred_price, label="Real Predicted")
            axes[1].set_title(f"Batch {i+1} – Real Prices")
            axes[1].set_xlabel("Prediction Step")
            axes[1].set_ylabel("Price")
            axes[1].legend()

            plt.tight_layout()
            
            # Save or show the plot
            if save_dir:
                plt.savefig(os.path.join(save_dir, f'validation_batch_{i+1}.png'))
                plt.close()
            else:
                plt.show()

            # Calculate metrics
            true = y_true.flatten()
            pred = y_pred.flatten()

            # --- Jaggedness metrics ---
            def mean_abs_diff(x):
                return np.mean(np.abs(np.diff(x)))
            
            pred_jagg = mean_abs_diff(pred)
            true_jagg = mean_abs_diff(true)
            jagg_ratio = pred_jagg / (true_jagg + 1e-8)  # avoid divide by zero
            
            metrics.append({
                'batch': i+1,
                'true_std': float(np.std(true)),
                'pred_std': float(np.std(pred)),
                'mse': float(np.mean((true - pred) ** 2)),
                'mae': float(np.mean(np.abs(true - pred))),
                'true_jaggedness': float(true_jagg),
                'pred_jaggedness': float(pred_jagg),
                'jaggedness_ratio': float(jagg_ratio)
            })
            
    return metrics

def save_experiment_results(config, train_hist, val_hist, steps_hist, model, val_loader_1, asset_idx, scaler, num_batches=10):
    """Save experimental results including losses, plots, and metrics"""
    # Create experiment directory with timestamp
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    exp_dir = f"experiments_{timestamp}"
    os.makedirs(exp_dir, exist_ok=True)
    
    # Calculate and save total learnable parameters
    total_learnable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    # Save configuration with model size
    config_with_params = config.copy()
    config_with_params['total_learnable_parameters'] = total_learnable_params
    with open(os.path.join(exp_dir, 'config.json'), 'w') as f:
        json.dump(config_with_params, f, indent=4)
    
    # Create and save loss plot and data
    fig, df_loss = create_loss_plot(train_hist, val_hist, steps_hist, total_learnable_params)
    fig.write_html(os.path.join(exp_dir, 'loss_plot.html'))
    df_loss.to_csv(os.path.join(exp_dir, 'loss_history.csv'))
    
    # Create validation plots and get metrics
    metrics = create_prediction_plots(
        model=model,
        val_loader=val_loader_1,
        asset_idx=asset_idx,
        scaler=scaler,
        config=config,
        num_batches=num_batches,
        save_dir=exp_dir
    )
    
    # Save metrics
    metrics_df = pd.DataFrame(metrics)
    metrics_df.to_csv(os.path.join(exp_dir, 'validation_metrics.csv'), index=False)
    
    # Save model summary information
    with open(os.path.join(exp_dir, 'model_summary.txt'), 'w') as f:
        f.write(f"Total Learnable Parameters: {total_learnable_params:,}\n")
        f.write(f"\nModel Configuration:\n")
        for key, value in config_with_params.items():
            f.write(f"{key}: {value}\n")
    
    return exp_dir

In [5]:
# -----------------------------
# Load and preprocess data
# -----------------------------
csv_path = r"D:\Quan\Quants\Neural Network\financial_attention\1h_data_20220101_20250601.csv"
closes = pd.read_csv(csv_path, index_col=0, parse_dates=True)[['SOL', 'ETH', 'BTC','ADA','XRP','LTC','TRX','LINK','DOT','DOGE']]



In [6]:
# Define parameter grid
param_grid = {
    'd_model': [32,64],  # Model dimensions
    'distill': [False],   # Whether to use distillation
    'use_time_embedding': [False, True],  # Whether to use time embeddings
    'dropout': [0.05,0.1]  # Dropout rates
}

# Generate all possible combinations
from itertools import product

# Generate all combinations
keys = param_grid.keys()
configs = []
for values in product(*param_grid.values()):
    config_dict = dict(zip(keys, values))
    # Base configuration
    config = {
        "d_input": len(closes.columns),
        "n_heads": 4,
        "enc_layers": 3,
        "dec_layers": 2,
        "enc_len": 96,
        "guiding_len": 48,
        "pred_len": 24,
        "factor": 5,
    }
    # Update with current combination
    config.update(config_dict)
    # Set d_ff to 4x d_model
    config['d_ff'] = config['d_model'] * 4
    configs.append(config)

print(f"Total number of configurations to test: {len(configs)}")
for i, cfg in enumerate(configs):
    print(f"\nConfiguration {i+1}:")
    print(f"d_model: {cfg['d_model']}, d_ff: {cfg['d_ff']}")
    print(f"distill: {cfg['distill']}, use_time_embedding: {cfg['use_time_embedding']}")
    print(f"dropout: {cfg['dropout']}")

Total number of configurations to test: 8

Configuration 1:
d_model: 32, d_ff: 128
distill: False, use_time_embedding: False
dropout: 0.05

Configuration 2:
d_model: 32, d_ff: 128
distill: False, use_time_embedding: False
dropout: 0.1

Configuration 3:
d_model: 32, d_ff: 128
distill: False, use_time_embedding: True
dropout: 0.05

Configuration 4:
d_model: 32, d_ff: 128
distill: False, use_time_embedding: True
dropout: 0.1

Configuration 5:
d_model: 64, d_ff: 256
distill: False, use_time_embedding: False
dropout: 0.05

Configuration 6:
d_model: 64, d_ff: 256
distill: False, use_time_embedding: False
dropout: 0.1

Configuration 7:
d_model: 64, d_ff: 256
distill: False, use_time_embedding: True
dropout: 0.05

Configuration 8:
d_model: 64, d_ff: 256
distill: False, use_time_embedding: True
dropout: 0.1


In [7]:
def run_experiment(config, exp_dir, exp_name):
    """Run a single experiment with given configuration"""
    # Set seeds for reproducibility
    set_seed(42)
    
    # Create data loaders
    train_loader, val_loader, scaler, asset_idx = create_dataloaders(
        closes, 
        enc_len=config["enc_len"],
        pred_len=config["pred_len"],
        batch_size=32,
        val_batch_size=32, 
        val_ratio=0.1, 
        asset_name="SOL"
    )
    
    # Initialize model
    model = InformerForecaster(config, asset_index=asset_idx)
    model.apply(init_weights)
    learnable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    # Training configuration
    tcfg = TrainConfig(
        learning_rate=1e-4,
        weight_decay=0.01,
        max_steps=10000,
        warmup_steps=200,
        use_amp=True,
        device="cuda",
        patience=15,
        min_delta=0.0001
    )
    
    # Train model
    model, train_hist, val_hist, steps_hist, best_val_loss, best_val_step = train_model(
        model, train_loader, val_loader, tcfg, asset_index=asset_idx
    )
    
    # Best validation step
    best_step = best_val_step if best_val_step is not None else (min(val_hist, key=lambda t: t[1])[0] if val_hist else float('nan'))
    
    # Create validation loader for visualization
    _, val_loader_1, _, _ = create_dataloaders(
        closes, 
        enc_len=config["enc_len"],
        pred_len=config["pred_len"],
        batch_size=32, 
        val_batch_size=1,
        val_shuffle=True,
        val_ratio=0.1, 
        asset_name="SOL"
    )
    
    # Create experiment-specific directory
    exp_subdir = os.path.join(exp_dir, exp_name)
    os.makedirs(exp_subdir, exist_ok=True)
    
    # Save model and configuration
    model_save_path = os.path.join(exp_subdir, 'model.pth')
    torch.save({
        'model_state_dict': model.state_dict(),
        'config': config,
        'total_params': learnable_params,
        'train_hist': train_hist,
        'val_hist': val_hist,
        'steps_hist': steps_hist
    }, model_save_path)
    
    # Create and save plots
    fig, df_loss = create_loss_plot(
        train_hist, 
        val_hist, 
        steps_hist,
        total_params=learnable_params
    )
    fig.write_html(os.path.join(exp_subdir, 'loss_plot.html'))
    df_loss.to_csv(os.path.join(exp_subdir, 'loss_history.csv'))
    
    # Calculate and save metrics
    metrics = create_prediction_plots(
        model=model,
        val_loader=val_loader_1,
        asset_idx=asset_idx,
        scaler=scaler,
        config=config,
        num_batches=10,
        save_dir=exp_subdir
    )
    
    metrics_df = pd.DataFrame(metrics)
    metrics_df.to_csv(os.path.join(exp_subdir, 'metrics.csv'), index=False)
    
    # Save summary metrics
    summary_metrics = metrics_df.mean().round(4)
    
    # Final train and best validation
    final_train_loss = train_hist[-1]
    best_val_loss = best_val_loss if best_val_loss is not None else (min(val_hist, key=lambda t: t[1])[1] if val_hist else float('nan'))
    
    return {
        'Experiment no': exp_name,
        'd_model': config['d_model'],
        'd_ff': config['d_ff'],
        'distill': config['distill'],
        'time embedding': config['use_time_embedding'],
        'dropout': config['dropout'],
        'learnable params': learnable_params,
        'Best Val Step': best_step,
        'train loss': final_train_loss,
        'best validation loss': best_val_loss,
        'jaggedness_ratio (pred/real)': float(summary_metrics['jaggedness_ratio']),
        'MSE': float(summary_metrics['mse']),
        'MAE': float(summary_metrics['mae'])
    }

In [8]:
# Create main experiments directory with timestamp
exp_dir = f"experiments_grid_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(exp_dir, exist_ok=True)

# Run all experiments
results = []
for i, config in enumerate(configs):
    print(f"\nRunning experiment {i+1}/{len(configs)}")
    print("Configuration:", config)
    
    # Create experiment name from parameters
    exp_name = (f"d{config['d_model']}_"
               f"{'dist' if config['distill'] else 'nodist'}_"
               f"{'time' if config['use_time_embedding'] else 'notime'}_"
               f"drop{config['dropout']}")
    
    # Run experiment
    try:
        result = run_experiment(config, exp_dir, exp_name)
        results.append(result)
        print(f"Experiment {exp_name} completed successfully")
        print(f"Best val loss: {result['best validation loss']:.6f}")
        print(f"MSE: {result['MSE']:.6f}")
    except Exception as e:
        print(f"Experiment {exp_name} failed with error: {str(e)}")
        continue

# Create results summary with specific column order
columns = [
    'Experiment no', 
    # Model Params
    'd_model', 'd_ff', 'distill', 'time embedding', 'dropout', 'learnable params',
    # Results
    'Best Val Step', 'train loss', 'best validation loss', 
    'jaggedness_ratio (pred/real)', 'MSE', 'MAE'
]

results_df = pd.DataFrame(results)[columns]

# Save results with proper formatting
results_df.to_csv(os.path.join(exp_dir, 'all_results.csv'), index=False, float_format='%.6f')

# Create summary visualizations
fig = px.scatter(results_df, 
                 x='best validation loss', 
                 y='MSE',
                 hover_data=columns,
                 title='Validation Loss vs MSE across experiments',
                 labels={'Experiment no': 'Experiment Name'})  # Update label
fig.write_html(os.path.join(exp_dir, 'results_scatter.html'))

# Save experiment configuration summary
config_summary = {
    'timestamp': datetime.now().strftime('%Y%m%d_%H%M%S'),
    'total_experiments': len(configs),
    'parameter_grid': param_grid,
    'base_config': {k: v for k, v in configs[0].items() if k not in param_grid},
    'experiment_names': [r['Experiment no'] for r in results]
}
with open(os.path.join(exp_dir, 'experiment_config.json'), 'w') as f:
    json.dump(config_summary, f, indent=4)

# Print best models by different metrics
print("\nBest models by validation loss:")
print(results_df.nsmallest(3, 'best validation loss')[columns])

print("\nBest models by MSE:")
print(results_df.nsmallest(3, 'MSE')[columns])

2025-10-24 17:38:29,275 | INFO | num decayed parameter tensors: 41, with 70,304 parameters
2025-10-24 17:38:29,276 | INFO | num non-decayed parameter tensors: 69, with 2,657 parameters
2025-10-24 17:38:29,276 | INFO | Using fused AdamW: True



Running experiment 1/8
Configuration: {'d_input': 10, 'n_heads': 4, 'enc_layers': 3, 'dec_layers': 2, 'enc_len': 96, 'guiding_len': 48, 'pred_len': 24, 'factor': 5, 'd_model': 32, 'distill': False, 'use_time_embedding': False, 'dropout': 0.05, 'd_ff': 128}


2025-10-24 17:38:30,467 | INFO | [Step     0] train_loss=0.912368 | lr=0.000e+00 | samples/s=88.2
2025-10-24 17:38:31,733 | INFO | [Step    10] train_loss=0.844396 | lr=5.000e-06 | samples/s=25.3
2025-10-24 17:38:32,010 | INFO | [Step    20] train_loss=0.776828 | lr=1.000e-05 | samples/s=115.5
2025-10-24 17:38:32,321 | INFO | [Step    30] train_loss=1.021419 | lr=1.500e-05 | samples/s=103.2
2025-10-24 17:38:32,623 | INFO | [Step    40] train_loss=1.271781 | lr=2.000e-05 | samples/s=106.4
2025-10-24 17:38:32,913 | INFO | [Step    50] train_loss=0.848913 | lr=2.500e-05 | samples/s=110.3
2025-10-24 17:38:33,197 | INFO | [Step    60] train_loss=1.136676 | lr=3.000e-05 | samples/s=113.1
2025-10-24 17:38:33,467 | INFO | [Step    70] train_loss=1.129706 | lr=3.500e-05 | samples/s=119.0
2025-10-24 17:38:33,753 | INFO | [Step    80] train_loss=0.863782 | lr=4.000e-05 | samples/s=112.3
2025-10-24 17:38:34,046 | INFO | [Step    90] train_loss=1.072885 | lr=4.500e-05 | samples/s=109.6
2025-10-24 1

Experiment d32_nodist_notime_drop0.05 completed successfully
Best val loss: 0.006879
MSE: 0.004200

Running experiment 2/8
Configuration: {'d_input': 10, 'n_heads': 4, 'enc_layers': 3, 'dec_layers': 2, 'enc_len': 96, 'guiding_len': 48, 'pred_len': 24, 'factor': 5, 'd_model': 32, 'distill': False, 'use_time_embedding': False, 'dropout': 0.1, 'd_ff': 128}


2025-10-24 17:43:07,030 | INFO | [Step    10] train_loss=0.844402 | lr=5.000e-06 | samples/s=24.5
2025-10-24 17:43:07,308 | INFO | [Step    20] train_loss=0.776781 | lr=1.000e-05 | samples/s=115.5
2025-10-24 17:43:07,604 | INFO | [Step    30] train_loss=1.021436 | lr=1.500e-05 | samples/s=108.5
2025-10-24 17:43:07,898 | INFO | [Step    40] train_loss=1.271784 | lr=2.000e-05 | samples/s=109.2
2025-10-24 17:43:08,187 | INFO | [Step    50] train_loss=0.848945 | lr=2.500e-05 | samples/s=111.0
2025-10-24 17:43:08,456 | INFO | [Step    60] train_loss=1.136697 | lr=3.000e-05 | samples/s=119.6
2025-10-24 17:43:08,719 | INFO | [Step    70] train_loss=1.129676 | lr=3.500e-05 | samples/s=122.1
2025-10-24 17:43:08,991 | INFO | [Step    80] train_loss=0.863861 | lr=4.000e-05 | samples/s=118.1
2025-10-24 17:43:09,280 | INFO | [Step    90] train_loss=1.072872 | lr=4.500e-05 | samples/s=111.1
2025-10-24 17:43:09,553 | INFO | [Step   100] train_loss=0.615546 | lr=5.000e-05 | samples/s=117.6
2025-10-24 

...

Experiment d64_nodist_notime_drop0.1 completed successfully
Best val loss: 0.004147
MSE: 0.006200

Running experiment 7/8
Configuration: {'d_input': 10, 'n_heads': 4, 'enc_layers': 3, 'dec_layers': 2, 'enc_len': 96, 'guiding_len': 48, 'pred_len': 24, 'factor': 5, 'd_model': 64, 'distill': False, 'use_time_embedding': True, 'dropout': 0.05, 'd_ff': 256}


2025-10-24 19:05:51,046 | INFO | [Step    10] train_loss=0.747723 | lr=5.000e-06 | samples/s=19.7
2025-10-24 19:05:51,372 | INFO | [Step    20] train_loss=0.891346 | lr=1.000e-05 | samples/s=98.4
2025-10-24 19:05:51,708 | INFO | [Step    30] train_loss=0.994815 | lr=1.500e-05 | samples/s=95.3
2025-10-24 19:05:52,059 | INFO | [Step    40] train_loss=0.942374 | lr=2.000e-05 | samples/s=91.7
2025-10-24 19:05:52,413 | INFO | [Step    50] train_loss=1.039745 | lr=2.500e-05 | samples/s=90.5
2025-10-24 19:05:52,765 | INFO | [Step    60] train_loss=1.100221 | lr=3.000e-05 | samples/s=91.0
2025-10-24 19:05:53,098 | INFO | [Step    70] train_loss=1.252216 | lr=3.500e-05 | samples/s=96.3
2025-10-24 19:05:53,439 | INFO | [Step    80] train_loss=0.956290 | lr=4.000e-05 | samples/s=94.3
2025-10-24 19:05:53,792 | INFO | [Step    90] train_loss=0.829002 | lr=4.500e-05 | samples/s=90.6
2025-10-24 19:05:54,124 | INFO | [Step   100] train_loss=0.842001 | lr=5.000e-05 | samples/s=96.4
2025-10-24 19:05:55,

Experiment d64_nodist_time_drop0.05 completed successfully
Best val loss: 0.004852
MSE: 0.005700

Running experiment 8/8
Configuration: {'d_input': 10, 'n_heads': 4, 'enc_layers': 3, 'dec_layers': 2, 'enc_len': 96, 'guiding_len': 48, 'pred_len': 24, 'factor': 5, 'd_model': 64, 'distill': False, 'use_time_embedding': True, 'dropout': 0.1, 'd_ff': 256}


2025-10-24 19:10:47,709 | INFO | [Step    10] train_loss=0.747667 | lr=5.000e-06 | samples/s=19.3
2025-10-24 19:10:48,078 | INFO | [Step    20] train_loss=0.891403 | lr=1.000e-05 | samples/s=86.9
2025-10-24 19:10:48,443 | INFO | [Step    30] train_loss=0.995043 | lr=1.500e-05 | samples/s=87.7
2025-10-24 19:10:48,800 | INFO | [Step    40] train_loss=0.942612 | lr=2.000e-05 | samples/s=90.1
2025-10-24 19:10:49,154 | INFO | [Step    50] train_loss=1.039680 | lr=2.500e-05 | samples/s=90.5
2025-10-24 19:10:49,519 | INFO | [Step    60] train_loss=1.100235 | lr=3.000e-05 | samples/s=88.0
2025-10-24 19:10:49,873 | INFO | [Step    70] train_loss=1.252163 | lr=3.500e-05 | samples/s=90.6
2025-10-24 19:10:50,207 | INFO | [Step    80] train_loss=0.956292 | lr=4.000e-05 | samples/s=96.1
2025-10-24 19:10:50,605 | INFO | [Step    90] train_loss=0.829168 | lr=4.500e-05 | samples/s=80.5
2025-10-24 19:10:50,947 | INFO | [Step   100] train_loss=0.841989 | lr=5.000e-05 | samples/s=93.7
2025-10-24 19:10:52,

Experiment d64_nodist_time_drop0.1 completed successfully
Best val loss: 0.005014
MSE: 0.005700

Best models by validation loss:
                Experiment no  d_model  d_ff  distill  time embedding  \
4  d64_nodist_notime_drop0.05       64   256    False           False   
5   d64_nodist_notime_drop0.1       64   256    False           False   
6    d64_nodist_time_drop0.05       64   256    False            True   

   dropout  learnable params  Best Val Step  train loss  best validation loss  \
4     0.05            285185           5600    0.004351              0.003669   
5     0.10            285185           5600    0.005158              0.004147   
6     0.05            291417           4500    0.006499              0.004852   

   jaggedness_ratio (pred/real)     MSE     MAE  
4                        2.1325  0.0062  0.0641  
5                        2.3044  0.0062  0.0645  
6                        2.5278  0.0057  0.0584  

Best models by MSE:
                Experiment no  d